# 📝 OpenAI API 활용 과제 LV3 정답 — 통합 프로그램 (강사용)

두 통합 과제의 **모범답안 + 해설**입니다. 자가채점은 각 단계 산출물의 **구조**만 검사합니다.

- 경로는 정답 노트북 기준 `../../day14_OpenAI_API_활용/data/` 입니다.

아래 준비 셀들을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 클라이언트 준비 — 이 셀을 먼저 실행하세요.
# .env 파일에 OPENAI_API_KEY 를 넣어 두면 아래 한 줄이 그것을 읽어 연결합니다.
#   참고: https://developers.openai.com/api/docs/guides/text
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('.env')                # 같은 폴더의 .env
load_dotenv('../../day14_OpenAI_API_활용/.env')             # (정답 폴더처럼 한 단계 안에서 열었을 때)

# max_retries: 분당 토큰 한도(TPM)에 걸리면(429) 잠시 뒤 자동으로 다시 시도한다.
#   이미지는 한 장에 수만 토큰이라 여러 장을 연달아 보내면 쉽게 걸린다.
client = OpenAI(max_retries=8)     # OPENAI_API_KEY 를 자동으로 찾아 쓴다
print('연결 준비 완료 —', '키 확인됨' if os.getenv('OPENAI_API_KEY') else '키가 없습니다(.env 를 확인하세요)')

In [ ]:
# [제공 코드] 라이브러리
import pandas as pd


In [ ]:
# [제공 코드] 감정분석 스키마 + analyze() (교안에서 만든 그대로 — 그냥 실행하세요)
import json
senti_schema = {'type': 'json_schema', 'json_schema': {
    'name': 'review_sentiment',
    'schema': {'type': 'object',
        'properties': {
            'sentiment': {'type': 'string', 'enum': ['긍정', '부정', '중립']},
            'summary': {'type': 'string'}},
        'required': ['sentiment', 'summary'], 'additionalProperties': False},
    'strict': True}}

def analyze(text):
    resp = client.chat.completions.create(model='gpt-4o-mini',
        messages=[{'role': 'system', 'content': '리뷰 감정을 분석해 스키마에 맞춰 답해.'},
                  {'role': 'user', 'content': str(text)}],
        response_format=senti_schema, temperature=0)
    return json.loads(resp.choices[0].message.content)

---
# 프로그램 1) 자세밴드 리뷰 감정분석 리포트

리뷰 데이터를 받아 **감정을 분석 → 집계 → 경영진용 리포트**까지 자동 생성하는 프로그램을 단계별로 완성합니다. 데이터는 `data/reviews.csv`(자세밴드 40건, 열: `review_id`·`product`·`rating`·`content`).

### 1단계 — 데이터 로드·살펴보기
`data/reviews.csv` 를 읽어 변수 **`reviews`** 에 담고, `shape`·`head()`·별점 분포를 확인하세요. (살펴보기 출력은 자유. 채점은 `reviews` 가 DataFrame 이고 40행인지만 봅니다.)

In [ ]:
reviews = pd.read_csv('../../day14_OpenAI_API_활용/data/reviews.csv')
# 분석에 들어가기 전에 크기·앞부분·분포를 먼저 본다 — 파일을 잘못 읽었다면 여기서 바로 드러난다.
print('크기:', reviews.shape)
display(reviews.head(3))
display(reviews['rating'].value_counts().sort_index().to_frame('개수'))

In [ ]:
# [자가채점]
assert isinstance(reviews, pd.DataFrame) and reviews.shape[0] == 40
print('✅ 통과!')

### 해설 — 1단계 — 데이터 로드·살펴보기

분석 전에 **몇 건인지·어떤 열인지·별점이 어떻게 퍼져 있는지**를 먼저 봅니다. 이 습관이 뒤 단계의 집계 결과가 말이 되는지 판단하는 기준이 됩니다.

### 2단계 — 배치 감정분석
제공된 `analyze()` 로 리뷰를 분석해 결과 딕셔너리 리스트 **`results`** 에 담으세요. (각 결과는 `sentiment`·`summary` 키를 가집니다.)

⚠️ **앞에서부터 자르지 마세요.** 이 CSV 는 **별점 오름차순으로 정렬**돼 있어 `head(12)` 로 자르면 1~2점 리뷰만 뽑혀 **감정 분포가 한쪽으로 쏠립니다**(3단계 집계가 의미를 잃습니다). **별점별로 고르게** 3건씩 뽑아 총 **15건**을 분석하세요 — `sample` 변수에 담고 그것을 도세요.

⚠️ 그리고 **한 건이 실패해도 멈추지 않게** 만드세요(교안 3교시 5절). 실제 데이터에는 빈 칸이 섞여 들어옵니다 — 그 상황을 직접 만들어 확인합니다.

- `sample['content']` 를 리스트로 만들고 **여덟 번째 항목(`[7]`)을 빈 문자열로 바꿔** 변수 **`texts_with_hole`** 에 담으세요(실제 데이터의 빈 칸을 흉내 낸 것입니다).
- 그 리스트를 돌며 분석하되, **내용이 없으면 호출하지 말고** 나머지는 `try/except` 로 감싸세요. 빈 자리·실패한 자리에는 **`None` 을 대신 담아** `results` 의 길이가 15로 유지되게 하세요.

**예시**
```
sample = reviews.groupby('rating', group_keys=False).head(3)   # 별점 1~5 × 3건 = 15건
len(results)  →  15
```
<details><summary>힌트</summary>

```text
세부구현:
1. groupby('rating') 후 각 그룹에서 head(3) 을 뽑아 sample 에 담는다(group_keys=False).
2. list(sample['content']) 로 만든 뒤 [7] 자리를 '' 로 바꾼다(변수 texts_with_hole).
3. 빈 리스트로 시작해(변수 results) texts_with_hole 을 돈다.
4. 내용이 없으면 None 을 담고 continue, 아니면 try/except 로 analyze 를 부른다(실패해도 None).
```

</details>

In [ ]:
sample = reviews.groupby('rating', group_keys=False).head(3)   # 별점별 3건 = 15건
print('표본 별점 분포:', sample['rating'].value_counts().sort_index().to_dict())

texts_with_hole = list(sample['content'])
texts_with_hole[7] = ''                           # 실제 데이터의 빈 칸을 흉내 낸다

results = []
for text in texts_with_hole:                      # 빈 칸이 하나 섞여 있다
    if not str(text).strip():                     # 내용이 없으면 부르지 않는다
        results.append(None)
        continue
    try:
        results.append(analyze(text))
    except Exception as e:                        # 한 건 실패가 전체를 멈추지 않게
        print('  실패:', type(e).__name__)
        results.append(None)
ok = [r for r in results if r]
print('분석 완료:', len(results), '건 (성공', len(ok), '· 건너뜀', len(results) - len(ok), ')')
print('예:', ok[0])

In [ ]:
# [자가채점]
assert len(results) == 15, 'results 길이는 sample 과 같아야 합니다(실패한 자리는 None 으로 남깁니다)'
assert results[7] is None, '빈 칸 자리는 None 이어야 합니다 — 한 건 실패가 전체를 멈추면 안 됩니다'
assert all(set(r.keys()) == {'sentiment', 'summary'} for r in results if r)
assert sum(1 for r in results if r) == 14, '나머지 14건은 정상 분석돼야 합니다'
# 별점별로 고르게 뽑았는지 — 한쪽 별점만 담겼으면 분포 집계가 무의미하다
assert dict(sample['rating'].value_counts().sort_index()) == {1: 3, 2: 3, 3: 3, 4: 3, 5: 3}
print('✅ 통과!')

### 해설 — 2단계 — 배치 감정분석

리뷰를 하나씩 돌며 **구조화된 출력**으로 감정을 받습니다. 자유 텍스트로 받으면 '긍정입니다'·'긍정적'처럼 표기가 흔들려 집계가 안 됩니다 — 스키마로 값을 고정해야 다음 단계에서 셀 수 있습니다.

### 3단계 — 집계
`results` 를 DataFrame 으로 만들어, 감정 분포를 센 딕셔너리 **`dist`**(예 `{'긍정':7,'부정':6,'중립':2}` — 합이 표본 15건)와 부정 리뷰들의 요약(`summary`) 리스트 **`neg_summaries`** 를 만드세요.

<details><summary>힌트</summary>

```text
세부구현:
1. results 로 DataFrame 을 만든다.
2. sentiment 열의 값별 개수를 세어 딕셔너리로 만든다(변수 dist).
3. sentiment 가 '부정'인 행의 summary 열만 리스트로 모은다(변수 neg_summaries).
```

</details>

In [ ]:
df = pd.DataFrame([r for r in results if r])      # 건너뛴 자리는 빼고 센다
dist = df['sentiment'].value_counts().to_dict()
neg_summaries = df[df['sentiment'] == '부정']['summary'].tolist()
print('감정 분포:', dist)
print('부정 요약 수:', len(neg_summaries))

In [ ]:
# [자가채점]
assert isinstance(dist, dict) and sum(dist.values()) == 14, '건너뛴 한 건을 빼고 14건이어야 합니다'
assert isinstance(neg_summaries, list)
# 별점을 고르게 뽑았으니 감정도 한 종류로 몰리면 안 된다 — 분포다운 분포인지 확인
assert len(dist) >= 2, '감정이 한 종류뿐이면 표본이 한쪽으로 쏠린 것입니다'
assert len(neg_summaries) > 0, '저평점 리뷰를 포함했다면 부정 요약이 있어야 합니다'
print('✅ 통과!')

### 해설 — 3단계 — 집계

2단계가 값을 고정해 준 덕분에 `Counter`·`value_counts` 로 바로 셀 수 있습니다. **구조화 출력이 집계를 가능하게 한다**는 것이 이 문제의 핵심 연결고리입니다.

### 4단계 — LLM 리포트 생성
집계 결과(`dist`)와 부정 요약(`neg_summaries`)을 프롬프트에 넣어, `gpt-4o-mini` 로 **경영진용 3줄 요약 리포트**를 생성해 문자열 **`report`** 에 담으세요. (프롬프트에 감정 분포 숫자와 부정 요약을 포함하고, '경영진이 읽을 3줄 리포트로' 라고 지시하세요.)

<details><summary>힌트</summary>

```text
세부구현:
1. content 프롬프트에 f-string 으로 dist 와 '\n'.join(neg_summaries) 를 넣는다.
2. '위 결과를 경영진용 3줄 리포트로 정리해줘' 를 덧붙여 create 호출.
3. 답 content 를 report 에 담는다.
```

</details>

In [ ]:
# 앞 단계에서 코드로 구한 값(dist·neg_summaries)을 프롬프트에 넣는다 —
#  집계는 코드가 하고 글쓰기만 모델에 맡기는 것이 숫자가 틀리지 않는 길이다.
prompt = (f'자세밴드 리뷰 감정 분포: {dist}\n'
          f'주요 부정 요약:\n' + '\n'.join(neg_summaries) + '\n\n'
          '위 결과를 경영진이 읽을 3줄 리포트로 정리해줘.')
resp = client.chat.completions.create(model='gpt-4o-mini',
    messages=[{'role': 'system', 'content': '너는 데이터 분석 리포트 작성자야. 간결한 한국어로 답해.'},
              {'role': 'user', 'content': prompt}],
    temperature=0.3)   # 글이라 0 보다는 조금 올리되, 사실이 흔들리지 않게 낮게 둔다
report = resp.choices[0].message.content
print(report)

In [ ]:
# [자가채점]
assert isinstance(report, str) and len(report.strip()) > 0
print('✅ 통과!')

### 해설

로드→분석→집계→생성으로 이어지는 파이프라인입니다. 구조화 출력(2단계)이 집계(3단계)를 가능케 하고, 그 숫자를 다시 LLM 에 넣어 사람이 읽을 리포트(4단계)를 만듭니다. 이것이 'AI 협업 인사이트 리포트'의 축소판입니다.

### 5단계 — 리포트를 파일로 저장
완성한 리포트는 파일로 남겨야 쓸모가 있습니다. 3단계의 감정 분포(`dist`)와 4단계의 리포트(`report`)를 하나의 텍스트로 묶어 **`output/sentiment_report.txt`** 에 저장하세요(4일차에서 배운 파일 쓰기).

- `output` 폴더가 없으면 만드세요(`os.makedirs('output', exist_ok=True)`).
- 저장한 파일 경로 문자열을 변수 **`report_path`** 에 담으세요.
- 파일 안에는 **감정 분포(`dist`)와 리포트 본문(`report`)이 둘 다** 들어가야 합니다(꾸미는 형식은 자유).

<details><summary>힌트</summary>

```text
세부구현:
1. os.makedirs('output', exist_ok=True) 로 폴더를 준비한다.
2. report_path 에 'output/sentiment_report.txt' 를 담는다.
3. with open(report_path, 'w', encoding='utf-8') 로 dist 와 report 를 이어 쓴다.
```

</details>

In [ ]:
import os
# exist_ok=True — 폴더가 이미 있어도 오류를 내지 않는다(다시 실행해도 안전하다).
os.makedirs('output', exist_ok=True)
report_path = 'output/sentiment_report.txt'
# 한글이 깨지지 않게 encoding 을 명시한다(맥·윈도우 기본값이 서로 다르다).
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('=== 감정 분포 ===\n')
    f.write(str(dist) + '\n\n')
    f.write('=== 리포트 ===\n')
    f.write(report)
print('저장 완료:', report_path)

In [ ]:
# [자가채점]
import os
assert os.path.exists(report_path)
saved = open(report_path, encoding='utf-8').read()
# 리포트 본문과 감정 분포가 둘 다 담겼는지 (꾸미는 형식은 자유)
assert report.strip() in saved, '4단계의 report 본문이 파일에 들어 있어야 합니다'
assert all(k in saved for k in dist), '감정 분포(dist)도 함께 저장해야 합니다'
print('✅ 통과!')

### 해설

분석 결과를 **파일로 남기는 것**까지가 실무 파이프라인입니다. 4일차의 `open(...,'w')` 파일 쓰기를 그대로 활용했습니다 — 이렇게 만든 리포트를 메일·슬랙으로 공유하거나 다음 단계 입력으로 씁니다.

---
# 프로그램 2) 선크림 리뷰 상담 어시스턴트

사용자의 질문에 따라 **알맞은 도구(함수)를 골라 실행**하고 답하는 어시스턴트를 만듭니다. 데이터는 `data/reviews_sun.csv`(선크림 20건). Function Calling 으로 **여러 도구 + 라우팅**을 구현합니다.

### 해설 — 문제 2 · 1단계 — 데이터·도구 함수 준비

먼저 **평범한 파이썬 함수**를 만듭니다. LLM 과 무관하게 그 자체로 동작해야 합니다 — 도구는 '모델이 부를 수 있게 설명을 붙인 우리 함수'일 뿐입니다.

### 1단계 — 데이터·도구 함수 준비
`data/reviews_sun.csv` 를 **`sun`** 에 담으세요. 도구로 쓸 함수 세 개는 아래 제공 셀에 있습니다(별점별 개수·키워드 개수·평균 별점). 실행만 하면 됩니다.

In [ ]:
sun = pd.read_csv('../../day14_OpenAI_API_활용/data/reviews_sun.csv')
print('선크림 리뷰:', sun.shape)
display(sun.head(2))

In [ ]:
# [자가채점]
assert isinstance(sun, pd.DataFrame) and sun.shape[0] == 20
print('✅ 통과!')

In [ ]:
# [제공 코드] 도구로 쓸 함수 세 개
def count_by_rating(rating):
    return int((sun['rating'] == rating).sum())

def count_keyword(keyword):
    return int(sun['content'].str.contains(keyword).sum())

def avg_rating():
    return round(float(sun['rating'].mean()), 2)

### 해설 — 문제 2 · 2단계 — 도구 스키마와 디스패처

스키마는 **모델이 읽을 설명서**이고, 디스패처는 **이름 → 실제 함수** 를 잇는 딕셔너리입니다. 도구가 늘어나도 `if` 를 쌓지 않고 딕셔너리에 한 줄 추가하면 되도록 만드는 것이 요령입니다.

### 2단계 — 도구 스키마와 디스패처
세 함수를 모델에게 알려 줄 `tools` 스키마 리스트 **`tools`** 를 만들고, 함수 이름을 실제 함수에 연결하는 **`dispatch`** 딕셔너리(`{'count_by_rating': count_by_rating, ...}`)를 만드세요.

- `count_by_rating`: 정수 인자 `rating`
- `count_keyword`: 문자열 인자 `keyword`
- `avg_rating`: 인자 없음(빈 properties)

<details><summary>힌트</summary>

```text
세부구현:
1. tools 리스트에 세 함수의 스키마를 각각 넣는다(avg_rating 은 properties 를 빈 딕셔너리로).
2. 함수 이름(문자열)을 키로, 실제 함수를 값으로 하는 딕셔너리를 만든다(변수 dispatch).
```

</details>

In [ ]:
tools = [
    {'type': 'function', 'function': {'name': 'count_by_rating',
        'description': '특정 별점(1~5) 리뷰 개수', 'parameters': {'type': 'object',
        'properties': {'rating': {'type': 'integer'}}, 'required': ['rating']}}},
    {'type': 'function', 'function': {'name': 'count_keyword',
        'description': '본문에 특정 키워드가 든 리뷰 개수', 'parameters': {'type': 'object',
        'properties': {'keyword': {'type': 'string'}}, 'required': ['keyword']}}},
    {'type': 'function', 'function': {'name': 'avg_rating',
        # 인자가 없는 도구는 properties 를 빈 딕셔너리로 두고 required 도 적지 않는다.
        'description': '전체 평균 별점', 'parameters': {'type': 'object', 'properties': {}}}}]

# 이름(문자열) → 실제 함수 표. 도구가 여럿이면 if 문을 늘리는 대신 이 표에서 찾아 부른다.
#  키는 위 tools 의 'name' 과 **글자 하나까지 같아야** 한다(다르면 KeyError).
dispatch = {'count_by_rating': count_by_rating,
            'count_keyword': count_keyword,
            'avg_rating': avg_rating}
print('도구', len(tools), '개 준비')

In [ ]:
# [자가채점]
assert len(tools) == 3
assert set(dispatch.keys()) == {'count_by_rating', 'count_keyword', 'avg_rating'}
print('✅ 통과!')

### 3단계 — 어시스턴트 함수
질문을 받아, 모델이 도구를 부르면 **디스패처로 실행 → 결과를 넣어 재호출 → 최종 답**을 하고, 도구가 필요 없으면 그냥 답하는 함수 **`ask_assistant(question)`** 를 만드세요(문자열을 반환).

<details><summary>힌트</summary>

```text
접근방법:
- 교안 2절의 4단계 흐름을 함수로 감싼다. tool_calls 유무로 분기한다.

세부구현:
1. 질문 하나로 messages 를 만들어 tools 를 넣어 1차 호출하고, 첫 메시지를 꺼낸다.
2. 그 메시지에 tool_calls 가 없으면 그대로 content 를 반환한다.
3. tool_calls 가 있으면 첫 호출의 함수 이름과 인자(json.loads)를 꺼내, dispatch 에서 그 이름의 함수를 찾아 인자로 실행한다.
4. assistant(tool_calls) 메시지와 tool(결과) 메시지를 붙여 2차 호출한 뒤 content 를 반환한다.
```

</details>

In [ ]:
def ask_assistant(question):
    messages = [{'role': 'user', 'content': question}]
    first = client.chat.completions.create(model='gpt-4o-mini', messages=messages, tools=tools)
    msg = first.choices[0].message
    # 도구가 필요 없는 질문이면 모델이 그냥 답한다 — 그때는 여기서 끝낸다.
    #  이 분기가 없으면 '팁 알려줘' 같은 질문에서 tool_calls[0] 이 None 이라 터진다.
    if not msg.tool_calls:
        return msg.content
    call = msg.tool_calls[0]
    args = json.loads(call.function.arguments)
    # 이름으로 표에서 찾아 부른다 — 도구를 늘려도 이 줄은 그대로다.
    result = dispatch[call.function.name](**args)
    messages.append({'role': 'assistant', 'content': None, 'tool_calls': [{'id': call.id,
        'type': 'function', 'function': {'name': call.function.name, 'arguments': call.function.arguments}}]})
    messages.append({'role': 'tool', 'tool_call_id': call.id, 'content': str(result)})
    second = client.chat.completions.create(model='gpt-4o-mini', messages=messages)
    return second.choices[0].message.content

print(ask_assistant('별점 5점 리뷰 몇 개야?'))

In [ ]:
# [자가채점]
reply = ask_assistant('별점 5점 리뷰 몇 개야?')
assert isinstance(reply, str) and len(reply.strip()) > 0
print('✅ 통과!')

### 해설 — 문제 2 · 3단계 — 어시스턴트 함수

교안 2절의 4단계 왕복을 함수 하나로 감쌉니다. **`tool_calls` 가 없으면 바로 답하고, 있으면 실행 후 다시 호출**하는 분기가 핵심입니다 — 이 분기를 빠뜨리면 일반 질문에서 에러가 납니다.

### 4단계 — 여러 질문으로 시험
아래 세 질문을 `ask_assistant` 로 처리해 답 문자열들을 리스트 **`answers`** 에 담으세요. 앞의 두 질문은 **도구가 필요하고**(평균 별점·키워드 개수), 마지막 질문은 도구 없이 모델이 바로 답합니다 — 세 답이 모두 나오면 라우팅이 제대로 도는 것입니다.

- 도구가 필요한 질문은 반드시 **`dispatch` 를 거쳐** 실제 함수가 실행돼야 합니다(자가채점이 이를 확인합니다).

```python
questions = ['평균 별점이 몇 점이야?', '백탁 언급한 리뷰 몇 개야?', '선크림 바를 때 팁 하나 알려줘.']
```

<details><summary>힌트</summary>

```text
세부구현:
1. 세 질문을 리스트로 만든다(변수 questions).
2. 각 질문을 ask_assistant 로 처리한 답들을 리스트로 모은다(변수 answers).
```

</details>

In [ ]:
# 앞의 둘은 도구가 필요하고, 마지막 하나는 모델이 그냥 답한다 —
#  같은 함수가 두 갈래를 모두 처리하는지 확인하는 것이 이 문제의 목적이다.
questions = ['평균 별점이 몇 점이야?', '백탁 언급한 리뷰 몇 개야?', '선크림 바를 때 팁 하나 알려줘.']
answers = [ask_assistant(q) for q in questions]
# zip 으로 짝지어 출력한다 — 질문과 답이 어긋나지 않는다.
for q, a in zip(questions, answers):
    print('Q:', q)
    print('A:', a, '\n')

In [ ]:
# [자가채점]
assert len(answers) == 3
assert all(isinstance(a, str) and len(a.strip()) > 0 for a in answers)

# 도구가 실제로 실행됐는지 — dispatch 의 함수를 잠깐 감싸 호출된 이름을 기록해 본다
called = []
real_fns = dict(dispatch)
def make_recorder(name, fn):
    def recorded(**kwargs):
        called.append(name)
        return fn(**kwargs)
    return recorded
for name, fn in real_fns.items():
    dispatch[name] = make_recorder(name, fn)
ask_assistant('평균 별점이 몇 점이야?')
dispatch.update(real_fns)          # 원래 함수로 되돌린다
assert 'avg_rating' in called, \
    '평균 별점 질문에는 avg_rating 도구가 dispatch 를 거쳐 실행돼야 합니다'
print('✅ 통과!')

### 해설

`ask_assistant` 는 **라우터** 역할입니다 — 데이터가 필요한 질문(별점·키워드·평균)엔 도구를 부르고, 일반 팁 질문엔 그냥 답합니다. 도구를 늘리려면 함수·스키마·`dispatch` 에 한 줄씩만 추가하면 됩니다. 이 구조가 다음 과목에서 배울 **에이전트**의 뼈대입니다.

---
# 프로그램 3) 영수증 정산 파이프라인

영수증 **사진**을 읽어 품목·금액 표로 만들고, **스스로 검산**한 뒤 파일로 남기는 프로그램입니다. 데이터는 `data/receipt/` 의 영수증 사진과, 사람이 정리해 둔 정답표 `data/receipts.csv` 입니다. (4교시의 이미지 정형화를 처음부터 끝까지 직접 만듭니다.)

### 해설 — 문제 3 · 1단계 — 스키마 설계

영수증은 **한 장에 품목이 여러 줄**이므로 중첩 구조가 필수입니다. 그리고 가게 이름은 사진에서 잘려 안 보일 수 있으므로 `Optional` 이어야 합니다 — 여기서 `str` 로 두면 모델이 가게 이름을 **지어냅니다**.

### 1단계 — 영수증 스키마 설계
영수증 한 장에서 받을 값을 pydantic 스키마로 **직접** 선언하세요.

- **`ReceiptItem`** : `name`(str) · `quantity`(int) · `price`(float, 그 줄의 합계 금액)
- **`Receipt`** : `store_name`(**Optional[str]**, 안 보이면 None) · `total_amount`(float) · `payment`(**Literal['현금','카드','기타']**) · `items`(**list[ReceiptItem]**)

<details><summary>힌트</summary>

```text
세부구현:
1. BaseModel 을 상속한 ReceiptItem 을 만든다(필드 3개).
2. Receipt 를 만들고 items 필드를 list[ReceiptItem] 으로 선언한다.
3. store_name 은 Optional[str] = Field(default=None, ...) 로 두고 description 에 '안 보이면 null' 을 적는다.
```

</details>

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

class ReceiptItem(BaseModel):
    name: str = Field(description='품목명. 영수증에 적힌 그대로')
    quantity: int = Field(description='수량. 안 적혀 있으면 1')
    # 금액은 float 로 받는다 — 나중에 합을 내 총액과 대조하려면 숫자여야 한다(문자열이면 못 더한다).
    price: float = Field(description='그 줄의 합계 금액')

class Receipt(BaseModel):
    # 흐릿하거나 잘린 영수증이 있으므로 가게 이름은 비어도 되게 둔다.
    store_name: Optional[str] = Field(default=None, description='가게 이름. 안 보이면 null')
    total_amount: float = Field(description='영수증에 적힌 총 결제 금액')
    payment: Literal['현금', '카드', '기타'] = Field(description='결제 수단')
    # 품목 수는 영수증마다 다르니 list 로 — 이 구조 덕분에 뒤에서 품목만 따로 표로 펼칠 수 있다.
    items: list[ReceiptItem] = Field(description='품목 목록')

print('필드:', list(Receipt.model_fields))

In [ ]:
# [자가채점]
fields = Receipt.model_fields
assert set(fields) == {'store_name', 'total_amount', 'payment', 'items'}
assert fields['store_name'].is_required() is False, 'store_name 은 Optional 이어야 합니다(안 보이면 None)'
assert set(ReceiptItem.model_fields) == {'name', 'quantity', 'price'}
print('✅ 통과!')

> 스키마를 직접 설계해 본 것으로 1단계는 끝입니다. **2단계부터는 아래 제공 셀의 스키마를 씁니다** — 뒤 단계의 채점 기준(품목 수·검산)이 이 스키마를 전제로 하기 때문입니다.

In [ ]:
# [제공 코드] 2단계부터 쓸 스키마 (이 셀을 실행하면 위에서 만든 것을 덮어씁니다)
from pydantic import BaseModel, Field
from typing import Literal, Optional

class ReceiptItem(BaseModel):
    name: str = Field(description='품목명. 영수증에 적힌 그대로')
    quantity: int = Field(description='수량. 안 적혀 있으면 1')
    price: float = Field(description='그 줄의 합계 금액')

class Receipt(BaseModel):
    store_name: Optional[str] = Field(default=None, description='가게 이름. 안 보이면 null')
    total_amount: float = Field(description='영수증에 적힌 총 결제 금액')
    payment: Literal['현금', '카드', '기타'] = Field(description='결제 수단')
    items: list[ReceiptItem] = Field(description='품목 목록')
print('제공 스키마로 진행합니다')

### 해설 — 문제 3 · 2단계 — 사진 배치 읽기

이미지를 넣는 방법은 4교시 1절 그대로입니다 — `content` 를 리스트로 만들고 `image_url` 에 data URL 을 넣습니다. 사진마다 품목이 여러 줄이므로 **영수증 표와 품목 표 두 개**로 나눠 담는 것이 뒤 단계(검산·저장)를 쉽게 만듭니다.

### 2단계 — 사진 3장 읽어 표 만들기
`../../day14_OpenAI_API_활용/data/receipt` 폴더의 사진을 이름순으로 정렬해 **앞 3장**을 읽으세요. 제공된 `to_data_url()` 을 씁니다.

- `parse`(`model='gpt-4o-mini'`, `response_format=Receipt`)로 각 사진을 분석하세요. 질문 텍스트는 **'이 영수증에서 가게·총액·결제수단·품목을 스키마대로 읽어줘. 안 보이는 값은 null.'** 로 하세요.
- 영수증 단위 정보를 `{'image_file': 파일명, 'store_name': ..., 'total_amount': ..., 'payment': ...}` 로 리스트 **`receipt_rows`** 에, 품목을 `{'image_file': 파일명, 'name': ..., 'price': ...}` 로 리스트 **`item_rows`** 에 담으세요.
- 각각 DataFrame **`receipt_df`**·**`item_df`** 로 만드세요.

<details><summary>힌트</summary>

```text
세부구현:
1. sorted(Path(...).glob('*.jpg'))[:3] 로 사진 3장을 고른다.
2. 사진마다 parse 를 호출하고 .parsed 를 꺼낸다.
3. 영수증 정보는 receipt_rows 에, r.items 를 돌며 품목은 item_rows 에 담는다(둘 다 image_file 을 함께).
4. pd.DataFrame 으로 각각 만든다.
```

</details>

In [ ]:
# [제공 코드] 이미지 → data URL 헬퍼 (교안에서 본 그대로)
import base64
from pathlib import Path

def to_data_url(path):
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    return f'data:image/jpeg;base64,{b64}'

In [ ]:
paths = sorted(Path('../../day14_OpenAI_API_활용/data/receipt').glob('*.jpg'))[:3]

# 영수증 한 장이 한 행(receipt), 품목 한 줄이 한 행(item) — 표를 둘로 나눠 담는다.
receipt_rows, item_rows = [], []
for p in paths:
    # 여기만 gpt-4o 다 — 작은 글씨·흐린 숫자를 읽어야 해서 mini 로는 자주 틀린다.
    resp = client.chat.completions.parse(model='gpt-4o',
        messages=[{'role': 'user', 'content': [
            {'type': 'text', 'text': '이 영수증에서 가게·총액·결제수단·품목을 스키마대로 읽어줘. 안 보이는 값은 null.'},
            {'type': 'image_url', 'image_url': {'url': to_data_url(p)}}]}],
        response_format=Receipt, max_tokens=1500)
    r = resp.choices[0].message.parsed
    receipt_rows.append({'image_file': p.name, 'store_name': r.store_name,
                         'total_amount': r.total_amount, 'payment': r.payment})
    for it in r.items:
        item_rows.append({'image_file': p.name, 'name': it.name, 'price': it.price})

receipt_df = pd.DataFrame(receipt_rows)
item_df = pd.DataFrame(item_rows)
display(receipt_df)
display(item_df)

In [ ]:
# [자가채점]
assert len(receipt_rows) == 3, '사진 3장을 모두 읽어야 합니다'
assert all({'image_file', 'store_name', 'total_amount', 'payment'} <= set(r.keys()) for r in receipt_rows), \
    '지문이 요구한 키가 다 있어야 합니다(item_count 같은 열을 더 담는 것은 괜찮습니다)'
assert len(item_rows) >= 3, '영수증 3장에서 품목이 최소 3줄은 나와야 합니다'
assert all({'image_file', 'name', 'price'} <= set(r.keys()) for r in item_rows), \
    '지문이 요구한 키가 다 있어야 합니다(quantity 같은 열을 더 담는 것은 괜찮습니다)'
assert item_df['image_file'].nunique() >= 2, '한 장만 처리하고 끝내지 않았는지 확인하세요'
print('✅ 통과!')

### 해설 — 문제 3 · 3단계 — 검산

**모델이 읽은 값을 그대로 믿지 않는 것**이 이 단계의 교훈입니다. 영수증은 스스로를 검산할 수단(품목 합 = 총액)을 갖고 있으므로, 그 관계로 자동 점검할 수 있습니다. 차이가 나는 영수증은 할인·부가세가 따로 적혔거나 숫자를 잘못 읽은 것이니 사람이 확인할 목록에 올립니다.

### 3단계 — 검산: 품목 합 == 총액
OCR 은 숫자를 잘못 읽습니다(0↔8, 1↔7). 영수증에는 검산 수단이 있습니다 — **품목 금액의 합이 총액과 같아야** 합니다.

- `item_df` 를 `image_file` 로 묶어 `price` 합을 구하고, `receipt_df` 의 `total_amount` 와 나란히 놓은 DataFrame **`verify_df`** 를 만드세요(열 이름은 자유).
- 두 값의 차이가 **1 미만이면 통과**로 보고, 통과한 영수증 수를 정수 **`ok_count`** 에 담으세요.
- 이어서 사람이 정리한 정답표(`receipts.csv`)의 `total_amount` 와도 대조해, **모델이 읽은 총액이 정답표와 맞는 장수**를 정수 **`match_count`** 에 담으세요(차이 1 미만이면 일치).

<details><summary>힌트</summary>

```text
세부구현:
1. item_df.groupby('image_file')['price'].sum() 으로 영수증별 품목 합을 구한다.
2. receipt_df 를 image_file 기준으로 그 합과 합친다(merge 또는 set_index+join).
3. (총액 - 품목합).abs() < 1 인 행 수를 세어 ok_count 에 담는다.
4. receipts.csv 를 읽어 image_file 로 merge 한 뒤, 총액 차이가 1 미만인 행 수를 match_count 에 담는다.
```

</details>

In [ ]:
# 검산 1 — 영수증 스스로와 대조한다(품목 합 vs 총액). 정답표가 없어도 할 수 있는 점검이다.
sums = item_df.groupby('image_file')['price'].sum().rename('item_sum')
verify_df = receipt_df.set_index('image_file')[['total_amount']].join(sums).fillna(0)
verify_df['diff'] = (verify_df['total_amount'] - verify_df['item_sum']).round(2)
# 정확히 0 이 아니라 '1원 미만'으로 본다 — 소수점 반올림 탓에 == 0 은 자주 빗나간다.
verify_df['ok'] = verify_df['diff'].abs() < 1
ok_count = int(verify_df['ok'].sum())
display(verify_df)
print('검산 통과:', ok_count, '/', len(verify_df), '장')

# 검산 2 — 사람이 적어 둔 정답표와 대조한다. 이쪽이 '맞았나'에 대한 진짜 답이다.
truth = pd.read_csv('../../day14_OpenAI_API_활용/data/receipts.csv')[['image_file', 'total_amount']]
# 열 이름이 양쪽 다 total_amount 라 suffixes 로 구분해 준다(안 주면 _x·_y 가 붙는다).
cmp = receipt_df[['image_file', 'total_amount']].merge(truth, on='image_file', suffixes=('_모델', '_정답'))
cmp['일치'] = (cmp['total_amount_모델'] - cmp['total_amount_정답']).abs() < 1
match_count = int(cmp['일치'].sum())
display(cmp)
print('정답표와 총액 일치:', match_count, '/', len(cmp), '장')

In [ ]:
# [자가채점]
assert verify_df.shape[0] == 3
assert isinstance(ok_count, int) and 0 <= ok_count <= 3
# 검산 기준이 실제로 계산됐는지 — 총액과 품목 합 두 값이 모두 표에 있어야 한다
assert verify_df.select_dtypes('number').shape[1] >= 2, '총액과 품목 합이 모두 들어 있어야 합니다'
# 금액이 전부 0 이면 영수증을 제대로 읽지 못한 것이다
assert float(verify_df.select_dtypes('number').abs().to_numpy().sum()) > 0, \
    '금액이 전부 0 입니다 — 2단계가 영수증을 제대로 읽었는지 확인하세요'
assert isinstance(match_count, int) and 0 <= match_count <= 3
assert set(['total_amount_모델', 'total_amount_정답']) <= set(cmp.columns), '정답표와 나란히 놓고 비교해야 합니다'
print('✅ 통과!  (검산 통과 수는 영수증에 따라 다릅니다 — 할인·부가세가 따로 적힌 영수증은 안 맞습니다)')

### 해설 — 문제 3 · 4단계 — 저장

LLM 호출 결과는 **반드시 저장**합니다. 같은 사진을 다시 돌리면 시간도 요금도 다시 듭니다. 엑셀에서 한글이 깨지지 않게 `encoding='utf-8-sig'` 를 씁니다.

### 4단계 — 결과 저장
`receipt_df` 를 **`output/receipts_parsed.csv`** 로, `item_df` 를 **`output/receipt_items_parsed.csv`** 로 저장하세요(둘 다 `index=False`, `encoding='utf-8-sig'`). 저장한 영수증 표 경로를 **`parsed_path`** 에 담으세요.

<details><summary>힌트</summary>

```text
세부구현:
1. os.makedirs('output', exist_ok=True) 로 폴더를 준비한다.
2. to_csv 로 두 파일을 저장한다(index=False, encoding='utf-8-sig').
3. 영수증 표 경로 문자열을 parsed_path 에 담는다.
```

</details>

In [ ]:
import os
os.makedirs('output', exist_ok=True)
parsed_path = 'output/receipts_parsed.csv'
# index=False — 저장할 때 인덱스를 빼야 다시 읽을 때 이름 없는 열이 생기지 않는다.
# utf-8-sig — 엑셀에서 열어도 한글이 깨지지 않게 하는 표시를 앞에 붙여 준다.
receipt_df.to_csv(parsed_path, index=False, encoding='utf-8-sig')
# 표가 둘이니 파일도 둘로 저장한다(영수증 단위 / 품목 단위).
item_df.to_csv('output/receipt_items_parsed.csv', index=False, encoding='utf-8-sig')
print('저장 완료:', parsed_path)

In [ ]:
# [자가채점]
import os
assert os.path.exists(parsed_path)
reloaded = pd.read_csv(parsed_path)
assert reloaded.shape[0] == 3, '영수증 3장이 그대로 저장돼야 합니다'
assert 'total_amount' in reloaded.columns
assert os.path.exists('output/receipt_items_parsed.csv'), '품목 표도 저장해야 합니다'
print('✅ 통과!')

---
수고했어요! 오늘 배운 대화·파라미터·프롬프트·Function Calling·구조화 출력·**이미지 정형화**를 묶어 **리포트 생성기**·**도구 쓰는 어시스턴트**·**영수증 정산기**를 완성했습니다. 이 경험이 다음 과목(RAG·에이전트)의 바탕이 됩니다.